# Model- and Data-Generated Grid Points
A module can choose to map itself to points on the grid by defining its set of points in an AMPL `.mod` model file or `.dat` data file.

In a `.dat` file, simply define the indexed set with its elements anywhere in the file.  The name of the module must also be added to the `DAT_MODULES` set, which maintains a partial list of active modules with grid points that will be added to the NLP.  Defining module grid points in a `.dat` is the easier of the two options.  A potential downside is having to enumerate every single grid point associated with the module, which can lead to a very large `.dat` file.

Defining grid points in a `.mod` file allows for more concise definitions.  However, having a module add its grid points in a `.mod` file requires strict adherence to AMPL's define-before-use rule, where the set has to be defined before it can be referred to in a model file.  Our workaround is to initially define an empty indexed set for the grid points of `MOD_MODULES` and then later in the model file redeclare `MOD_MODULES` with the actual grid points.

# Module Definitions
The following sets keep track of the active modules used to define the grid points in the NLP.

`MOD_MODULES`: The set of modules whose grid points that are created in a `.mod` file.

`MOD_GRID`:  An indexed set of grid points whose indices are modules in `MOD_MODULES`.

`DAT_MODULES`: The set of modules whose grid points that are created in a `.dat` file.

`DAT_GRID`: An indexed set of grid points whose indices are modules in `DAT_MODULES`.

`MODULES`: The union of `MOD_MODULES` and `DAT_MODULES`.

`GRID_MODULES`: The combined indexed set of grid points of `MOD_MODULES` and `DAT_MODULES`.

# Example
In the following example, we demonstrate how the model- and data-module code adds grid points to the NLP from 4 different modules.  Two of the modules are Amplify modules and the other two are meant to mimic modules external to the Amplify library.

In [ ]:
from amplpy import AMPL
ampl = AMPL()

In [ ]:
%%ampl_eval

#----------- modules
# model-defined modules
set MOD_MODULES;
set MOD_GRID {MOD_MODULES}; # this set will be redeclared later

# data-defined modules
set DAT_MODULES default {};
set DAT_GRID {DAT_MODULES};

# all modules
set MODULES = MOD_MODULES union DAT_MODULES;
set GRID_MODULES {i1 in MODULES} := 
  if i1 in MOD_MODULES then MOD_GRID[i1]  
  else DAT_GRID[i1];

In [ ]:
%%ampl_eval

# model-defined modules (e.g., Amplify built-ins)
#----------- defined in Runge-Kutta module
set ODE_GRID := 1..50;
#----------- defined in Equations of Motion module
set PIM dimen 2;
#----------- defined in Bezier module
set BEZ_LBLS;
set BEZ_SUBGRID {BEZ_LBLS};
set BEZ_SUBGRID_MBRS := {'BEG', 'END'};
param bez_subgrid {i1 in BEZ_LBLS, BEZ_SUBGRID[i1], BEZ_SUBGRID_MBRS};

# data-defined modules (e.g., user or external developer modules)
#----------- see data section in the cell below.

In [ ]:
%%ampl_eval

# model has been processed, so now add data specific rules for modules in the model file
#----------- the following is expected to be reserved for Amplify builtin modules
#----------- the basic idea is that it makes more sense to programmatically define
#----------- the sets in the model than in the data section; the assignments can be
#----------- with pre-defined sets like ODE_GRID or built up from other sources, see
#----------- in PIM and BEZ code in the redeclare statement.
redeclare set MOD_GRID {i1 in MOD_MODULES} := 
  if i1 = 'ODE' then ODE_GRID
  else if i1 = 'PIM' then union {(i2,i3) in PIM} {i2, i3}
  else if i1 = 'BEZ' then union {i2 in BEZ_LBLS, i3 in BEZ_SUBGRID[i2]} bez_subgrid[i2, i3, 'BEG']..bez_subgrid[i2, i3, 'END']
  else {};

In [ ]:
%%ampl_eval

data;
set PIM (10, 11) (20, 63);

# activate model-defined modules in the model file
#----------- notice that BEZ is not active
set MOD_MODULES := ODE PIM;

# activate data-defined modules in this data file
set DAT_MODULES = DAT_GRID_1 DAT_GRID_2;

#----------- data-defined sets for each data-defined module
set DAT_GRID[DAT_GRID_1] := 10 11 12 13 14 15 16 17;
set DAT_GRID[DAT_GRID_2] := 19 20 21 22 97 98 99 100;

model;

In [ ]:
%%ampl_eval

display DAT_MODULES, MOD_MODULES;
display DAT_GRID, MOD_GRID;
display GRID_MODULES;

set DAT_MODULES := DAT_GRID_1 DAT_GRID_2;

set MOD_MODULES := ODE PIM;

set DAT_GRID[DAT_GRID_1] := 10 11 12 13 14 15 16 17;
set DAT_GRID[DAT_GRID_2] := 19 20 21 22 97 98 99 100;

set MOD_GRID[ODE] :=
1    5    9    13   17   21   25   29   33   37   41   45   49
2    6    10   14   18   22   26   30   34   38   42   46   50
3    7    11   15   19   23   27   31   35   39   43   47
4    8    12   16   20   24   28   32   36   40   44   48;
set MOD_GRID[PIM] := 10 11 20 63;

set GRID_MODULES[ODE] :=
1    5    9    13   17   21   25   29   33   37   41   45   49
2    6    10   14   18   22   26   30   34   38   42   46   50
3    7    11   15   19   23   27   31   35   39   43   47
4    8    12   16   20   24   28   32   36   40   44   48;
set GRID_MODULES[PIM] := 10 11 20 63;
set GRID_MODULES[DAT_GRID_1] := 10 11 12 13 14 15 16 17;
set GRID_MODULES[DAT_GRID_2] := 19 20 21 22 97 98 99 100;

